In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.\
            builder.\
                appName('Spark DAG Plan').\
                    getOrCreate()

In [0]:
# Check default parallelism
# sparkContext is not available on serverless; check the parallelism config instead
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

In [0]:
df1 = spark.range(4, 200, 2)
df2 = spark.range(2, 200, 4)

df1.show(5)
df2.show(5)

In [0]:
from pyspark.sql.functions import spark_partition_id, col, countDistinct

# repartition the dataframes
df1 = df1.repartition(5).\
            withColumn('partition_id', spark_partition_id())

df2 = df2.repartition(7).\
            withColumn('partition_id', spark_partition_id())

df1_part = df1.select(countDistinct('partition_id')).collect()[0][0]
df2_part = df2.select(countDistinct('partition_id')).collect()[0][0]

print(df1_part, df2_part, sep='---')

In [0]:
# join the dataFrame
df = df1.join(df2, on='id')

# sum 
df_sum = df.selectExpr('sum(id) AS totalSum')
df_sum.show()

In [0]:
# Explain Plan
# read this bottom to up
df_sum.explain()

In [0]:
# skip jobs
df2 = df2.drop("partition_id")
df_union = df_sum.union(df2)
df_union.show()

In [0]:
df_union.explain()

In [0]:
# converting dataFrame in rdd
# df1.rdd

In [0]:
df_sum.explain(True)